# Feature Engineering Pipeline for Vertex AI

## 🎯 For Stakeholders: Simple Usage Guide

**You only need to do ONE thing: Edit `config.yaml` with your settings!**

This notebook automatically:
1. ✅ Loads your `config.yaml` file
2. ✅ Builds the training package as tar.gz
3. ✅ Uploads it to GCS automatically
4. ✅ Converts all config values to pipeline arguments
5. ✅ Creates and runs a Vertex AI Custom Training Job

### Quick Start:
1. **Edit `config.yaml`** - Update your project settings, SQL queries, model parameters, etc.
2. **Run all cells** - Execute the notebook cells in order (Cell → Run All)
3. **Done!** - The pipeline will run automatically with your configuration

### What you can configure in `config.yaml`:
- GCP project settings (project, dataset, bucket, etc.)
- BigQuery SQL queries for training/test data
- Data processing parameters (target column, variance threshold, etc.)
- Feature engineering settings (one-hot encoding, feature classification)
- Model configuration (XGBoost parameters)
- Output settings (file prefixes, result locations)

**Note:** All configuration is automatically passed to the training job - no need to upload config.yaml separately!


In [17]:
# Import necessary libraries
import os
import json
from datetime import datetime
from google.cloud import aiplatform
from kfp.v2 import dsl, compiler

# Import helper functions (handles package building, uploading, etc.)
from pipeline_helpers import (
    build_and_upload_package,
    create_custom_training_job_component
)

# Import config loader
from trainer.config import load_config


## Step 1: Load Your Configuration

The notebook will automatically load your `config.yaml` file and set up everything needed for the pipeline.


In [18]:
# Load your configuration from config.yaml
config = load_config("config.yaml")

# Setup pipeline constants (automatically extracted from config.yaml)
# All values come from config.yaml - no hardcoded values!
constants = {
    "PROJECT": config['gcp']['project'],
    "COMPUTE_PROJECT": config['gcp']['project'],
    "LOCATION": config['vertex_ai']['location'],
    "CMEK_KEY": config['vertex_ai']['cmek_key'],
    "SERVICE_ACCOUNT": config['vertex_ai']['service_account'],
    "DOCKER_URI": config['vertex_ai']['docker_uri'],
    "LOB": config['bigquery_labels']['lob'],
    "SHARED_PROJECT": config['gcp']['gcp_project'],
    "LABELS": {
        'owner': config['bigquery_labels']['owner'],
        'costcenter': config['bigquery_labels']['costcenter'],
        'tenant': 'hcm-cm-de',
        'self_serve': 'true',
        'lob': config['bigquery_labels']['lob'],
        'pipeline_type': config['bigquery_labels']['pipeline_type']
    }
}

print("✅ Configuration loaded from config.yaml")
print(f"   📍 Vertex AI Region: {constants['LOCATION']}")
print(f"   🐳 Docker Image: {constants['DOCKER_URI']}")


Loading config from: config.yaml
✅ Configuration loaded from config.yaml
   📍 Vertex AI Region: us-east4
   🐳 Docker Image: us-docker.pkg.dev/vertex-ai/training/xgboost-cpu.2-1:latest


## Step 2: Build and Upload Package

The notebook will automatically build your training package and upload it to Google Cloud Storage.


In [19]:
# Build and upload the training package
# This happens automatically - you don't need to modify anything here
# Pipeline root is read from config.yaml
pipeline_root = config['vertex_ai']['pipeline_root']
package_uri, package_file = build_and_upload_package(
    training_dir=".",
    output_dir="dist",
    project=constants["PROJECT"],
    pipeline_root=pipeline_root
)

print(f"\n✅ Package built and uploaded successfully!")
print(f"📦 Package location: {package_uri}")


Building package from ....
STDOUT: running sdist
running egg_info
creating feature_engineering_trainer.egg-info
writing feature_engineering_trainer.egg-info\PKG-INFO
writing dependency_links to feature_engineering_trainer.egg-info\dependency_links.txt
writing requirements to feature_engineering_trainer.egg-info\requires.txt
writing top-level names to feature_engineering_trainer.egg-info\top_level.txt
writing manifest file 'feature_engineering_trainer.egg-info\SOURCES.txt'
reading manifest file 'feature_engineering_trainer.egg-info\SOURCES.txt'
writing manifest file 'feature_engineering_trainer.egg-info\SOURCES.txt'
running check
creating feature_engineering_trainer-0.1.0
creating feature_engineering_trainer-0.1.0\feature_engineering_trainer.egg-info
creating feature_engineering_trainer-0.1.0\trainer
creating feature_engineering_trainer-0.1.0\utils
copying files to feature_engineering_trainer-0.1.0...
copying README.md -> feature_engineering_trainer-0.1.0
copying setup.py -> feature_eng

## Step 3: Create and Run Pipeline

The pipeline will be created and submitted to Vertex AI with all your configuration settings.


In [20]:
@dsl.pipeline(
    name="feature-engineering-pipeline",
    description="Pipeline to run feature engineering using custom training package. All config values are passed as arguments."
)
def feature_engineering_pipeline(
    project_id: str = "",
    region: str = "",
    cmek_key: str = "",
    pipeline_root: str = "",
    package_uri: str = "",
    python_module: str = "trainer.task",
    service_account: str = "",
    docker_uri: str = "",
    lob: str = "",
    shared_project: str = "",
    gcp_gcp_project: str = "",
    gcp_db: str = "",
    prefix: str = "",
    default_exp: str = "",
    sdoh_year: str = "",
    location: str = "",
    bucket_name: str = "",
    gcs_destination_path: str = "",
    owner: str = "",
    costcenter: str = "",
    unique_id: str = "",
    pipeline_type: str = "",
    expiration_days: int = 7,
    sql_queries: str = "",
    random_seed: int = 35,
    numpy_seed: int = 35,
    embedding_pattern: str = "",
    variance_threshold: float = 0.001,
    target_column: str = "",
    exclude_for_variance: str = "[]",
    exclude_for_classification: str = "[]",
    feature_classification: str = "",
    min_occurrence: int = 2000,
    model_config: str = "",
    undersampling_config: str = "",
    rfecv_config: str = "",
    metrics_config: str = "",
    output_config: str = "",
    bigquery_query_config: str = "",
    machine_type_param: str = "n1-standard-4",
    accelerator_type: str = None,
    accelerator_count: int = 1,
):
    """Feature engineering pipeline using custom training package.
    
    All configuration values from config.yaml are automatically passed as pipeline arguments.
    """
    # Reconstruct constants dict from parameters
    pipeline_constants = {
        "PROJECT": project_id,
        "COMPUTE_PROJECT": project_id,
        "LOCATION": region,
        "CMEK_KEY": cmek_key,
        "SERVICE_ACCOUNT": service_account,
        "DOCKER_URI": docker_uri,
        "LOB": lob,
        "SHARED_PROJECT": shared_project,
        "LABELS": {
            "owner": owner,
            "costcenter": costcenter,
            "tenant": "hcm-cm-de",
            "self_serve": "true",
            "lob": lob,
            "pipeline_type": pipeline_type
        }
    }
    
    # Set up environment variables
    env = {
        "GOOGLE_CLOUD_PROJECT": project_id,
        "GCP_PROJECT": project_id,
    }
    
    # Machine type configuration (from config.yaml parameters)
    machine_type = {
        "machineType": machine_type_param,
    }
    
    # Add accelerator configuration if specified
    if accelerator_type.value and accelerator_type.value != 'None' and accelerator_type != "null":  # Only add accelerator if type is specified
        machine_type["acceleratorType"] = accelerator_type
        machine_type["acceleratorCount"] = accelerator_count
    
    # Build command-line arguments for the training job
    training_args = [
        "--gcp-project", project_id,
        "--gcp-gcp-project", gcp_gcp_project,
        "--gcp-db", gcp_db,
        "--prefix", prefix,
        "--default-exp", default_exp,
        "--sdoh-year", sdoh_year,
        "--location", location,
        "--bucket-name", bucket_name,
        "--gcs-destination-path", gcs_destination_path,
        "--owner", owner,
        "--costcenter", costcenter,
        "--unique-id", unique_id,
        "--pipeline-type", pipeline_type,
        "--lob", lob,
        "--expiration-days", str(expiration_days),
        "--sql-queries", sql_queries,
        "--random-seed", str(random_seed),
        "--numpy-seed", str(numpy_seed),
        "--embedding-pattern", embedding_pattern,
        "--variance-threshold", str(variance_threshold),
        "--target-column", target_column,
        "--exclude-for-variance", exclude_for_variance,
        "--exclude-for-classification", exclude_for_classification,
        "--feature-classification", feature_classification,
        "--min-occurrence", str(min_occurrence),
        "--model-config", model_config,
        "--undersampling-config", undersampling_config,
        "--rfecv-config", rfecv_config,
        "--metrics-config", metrics_config,
        "--output-config", output_config,
        "--bigquery-query-config", bigquery_query_config,
    ]
    print(machine_type)
    print(accelerator_type)
    # Create custom training job component
    feature_engineering_job = create_custom_training_job_component(
        pipeline_root=pipeline_root,
        constants=pipeline_constants,
        machine_type=machine_type,
        package_uris=[package_uri],
        python_module=python_module,
        display_name="feature-engineering-training",
        env=env,
        args=training_args,
    )

print("✅ Pipeline defined")

{'machineType': {{channel:task=;name=machine_type_param;type=String;}}}
{{channel:task=;name=accelerator_type;type=String;}}
✅ Pipeline defined


In [21]:
# Compile and create the pipeline job
# All your config.yaml settings are automatically converted and passed to the pipeline

# Compile the pipeline
compiler.Compiler().compile(
    pipeline_func=feature_engineering_pipeline,
    package_path="feature_engineering_pipeline.json"
)
print("✅ Pipeline compiled")

# Initialize Vertex AI
aiplatform.init(
    project=constants["PROJECT"],
    location=constants["LOCATION"],
)

# Convert config.yaml values to JSON strings, then base64 encode them
# Base64 encoding prevents JSON escaping issues when passed as command-line arguments
import base64

def encode_json_arg(value):
    """Encode a JSON-serializable value as base64 string."""
    json_str = json.dumps(value, ensure_ascii=False, separators=(',', ':'))
    encoded = base64.b64encode(json_str.encode('utf-8')).decode('utf-8')
    return f'base64:{encoded}'

sql_queries_json = encode_json_arg(config['sql_queries'])
feature_classification_json = encode_json_arg(config['feature_classification'])
model_config_json = encode_json_arg(config['model'])
undersampling_config_json = encode_json_arg(config['undersampling'])
rfecv_config_json = encode_json_arg(config['rfecv'])
metrics_config_json = encode_json_arg(config['metrics'])
output_config_json = encode_json_arg(config['output'])
bigquery_query_config_json = encode_json_arg(config['bigquery_query'])
exclude_for_variance_json = encode_json_arg(config['data_processing']['exclude_for_variance'])
exclude_for_classification_json = encode_json_arg(config['data_processing']['exclude_for_classification'])

# Create pipeline job with all config.yaml settings
    # Convert empty string to None for accelerator_type
    # Convert empty string to None for accelerator_type
# Convert empty string or YAML None to Python None for accelerator_type
accelerator_type_value = config['vertex_ai'].get('accelerator_type', None)
# Handle YAML None (which might be loaded as the string "None" or actual None)
if accelerator_type_value is None:
    accelerator_type_value = None
elif isinstance(accelerator_type_value, str):
    accelerator_type_value = accelerator_type_value.strip()
    if not accelerator_type_value or accelerator_type_value.lower() in ("null", "none", ""):
        accelerator_type_value = None


pipeline_job = aiplatform.PipelineJob(
    display_name=f"feature-engineering-pipeline-{datetime.now().strftime('%Y%m%d%H%M%S')}",
    template_path="feature_engineering_pipeline.json",
    pipeline_root=pipeline_root,
    enable_caching=False,
    encryption_spec_key_name=constants["CMEK_KEY"],
    # Convert empty string to None for accelerator_type

    parameter_values={
        # Pipeline infrastructure (from config.yaml vertex_ai section)
        "project_id": constants["PROJECT"],
        "region": constants["LOCATION"],
        "cmek_key": constants["CMEK_KEY"],
        "pipeline_root": pipeline_root,
        "package_uri": package_uri,
        "python_module": "trainer.task",
        "service_account": constants["SERVICE_ACCOUNT"],
        "docker_uri": constants["DOCKER_URI"],
        "lob": constants["LOB"],
        "shared_project": constants["SHARED_PROJECT"],
        # Machine and Accelerator Configuration (from config.yaml)
        # Convert empty string to None for accelerator_type
        "machine_type_param": config['vertex_ai'].get('machine_type', 'n1-standard-4'),
        "accelerator_type": accelerator_type_value,
        "accelerator_count": config['vertex_ai'].get('accelerator_count', 1),
        "gcp_gcp_project": config['gcp']['gcp_project'],
        "gcp_db": config['gcp']['gcp_db'],
        "prefix": config['gcp']['prefix'],
        "default_exp": config['gcp']['default_exp'],
        "sdoh_year": config['gcp']['sdoh_year'],
        "location": config['gcp']['location'],
        "bucket_name": config['gcp']['bucket_name'],
        "gcs_destination_path": config['gcp']['gcs_destination_path'],
        # BigQuery Labels (from config.yaml)
        "owner": config['bigquery_labels']['owner'],
        "costcenter": config['bigquery_labels']['costcenter'],
        "unique_id": config['bigquery_labels']['unique_id'],
        "pipeline_type": config['bigquery_labels']['pipeline_type'],
        "expiration_days": config['bigquery_labels']['expiration_days'],
        # SQL Queries (as JSON string from config.yaml)
        "sql_queries": sql_queries_json,
        # Data Processing (from config.yaml)
        "random_seed": config['data_processing']['random_seed'],
        "numpy_seed": config['data_processing']['numpy_seed'],
        "embedding_pattern": config['data_processing']['embedding_pattern'],
        "variance_threshold": config['data_processing']['variance_threshold'],
        "target_column": config['data_processing']['target_column'],
        "exclude_for_variance": exclude_for_variance_json,
        "exclude_for_classification": exclude_for_classification_json,
        # Feature Classification (as JSON string from config.yaml)
        "feature_classification": feature_classification_json,
        # One-Hot Encoding (from config.yaml)
        "min_occurrence": config['one_hot_encoding']['min_occurrence'],
        # Model Configuration (as JSON string from config.yaml)
        "model_config": model_config_json,
        # Undersampling Configuration (as JSON string from config.yaml)
        "undersampling_config": undersampling_config_json,
        # RFECV Configuration (as JSON string from config.yaml)
        "rfecv_config": rfecv_config_json,
        # Metrics Configuration (as JSON string from config.yaml)
        "metrics_config": metrics_config_json,
        # Output Configuration (as JSON string from config.yaml)
        "output_config": output_config_json,
        # BigQuery Query Configuration (as JSON string from config.yaml)
        "bigquery_query_config": bigquery_query_config_json,
    },
    labels=constants["LABELS"]
)

print(f"\n✅ Pipeline job created successfully!")
print(f"📋 All settings from config.yaml have been loaded")
print(f"📦 Package: {package_uri}")

✅ Pipeline compiled

✅ Pipeline job created successfully!
📋 All settings from config.yaml have been loaded
📦 Package: gs://hcm-cm-de-code-hcb-dev/vertex-test/packages/feature_engineering_trainer-0.1.0.tar.gz


## Step 4: Run the Pipeline

Execute the pipeline on Vertex AI. This will take some time - you can monitor progress in the Vertex AI console.


In [ ]:
pipeline_job.submit(
    service_account=constants["SERVICE_ACCOUNT"],
)

print("\nPipeline submitted successfully!")
print(f"Track progress in the Vertex AI Console: {pipeline_job._dashboard_uri()}")


Creating PipelineJob
PipelineJob created. Resource name: projects/46378383599/locations/us-east4/pipelineJobs/feature-engineering-pipeline-20260326084904
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/46378383599/locations/us-east4/pipelineJobs/feature-engineering-pipeline-20260326084904')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/feature-engineering-pipeline-20260326084904?project=46378383599

✅ Pipeline submitted successfully!
🔗 Console URL: https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/feature-engineering-pipeline-20260326084904?project=46378383599

👉 Track progress in the Vertex AI Console:
